# Import packages
run the following cell to import necessary packages to run the analysis

In [ ]:
import os
from pathlib import Path
from tqdm.notebook import tqdm
import numpy as np
from matplotlib import pyplot as plt
from matplotlib.colors import ListedColormap

from ome_zarr.io import parse_url
from ome_zarr.reader import Reader, Node
from ome_zarr.utils import info
import napari
from napari.utils.colormaps import AVAILABLE_COLORMAPS

# Functions

In [ ]:
def crop_image(image_data, start_roi, end_roi):
    assert len(start_roi) == len(end_roi), "The defined ROIs don't have the same number of dimensions."
    assert len(start_roi) >= 3, "Too many dimensions on the ROI."
    
    if len(start_roi) == 3:
        cropped = image_data[..., start_roi[0]:end_roi[0], start_roi[1]:end_roi[1], start_roi[2]:end_roi[2] ]
    elif len(start_roi) == 2:
        cropped = image_data[..., start_roi[0]:end_roi[0], start_roi[1]:end_roi[1]]
    
    return cropped

def max_z_projection(image_data):
    assert len(image_data.shape) >=3, "Not 3D data."

    return np.max(image_data, axis=1)

def create_rgb_composite(image_data, percentile_norm=(0, 95)):
    """
    Create RGB composite using napari colormaps (magenta, green, bop blue).
    
    Parameters:
    - image_data: array with shape (3, height, width)
    - percentile_norm: contrast stretching percentiles
    
    Returns:
    - RGB array with shape (height, width, 3)
    """
    magenta_cmap = AVAILABLE_COLORMAPS['magenta']
    green_cmap = AVAILABLE_COLORMAPS['green']
    bop_blue_cmap = AVAILABLE_COLORMAPS['bop blue']
    
    colormaps = [magenta_cmap, green_cmap, bop_blue_cmap]
    
    rgb_composite = np.zeros((image_data.shape[1], image_data.shape[2], 3))
    
    for ch_idx, cmap in enumerate(colormaps):
        if ch_idx == 1:
            continue  # skip green channel for now
        channel = image_data[ch_idx]
        # Normalize to 0-1 range
        vmin = np.percentile(channel.flatten(), percentile_norm[0])
        vmax = np.percentile(channel.flatten(), percentile_norm[1])
        normalized = np.clip((channel - vmin) / (vmax - vmin), 0, 1)
        
        # Apply colormap and get RGBA values
        rgba = cmap.map(normalized)  # shape: (height, width, 4)
        
        # Add RGB components (ignore alpha channel)
        rgb_composite += rgba[..., :3]
    
    # Clip to valid range after blending
    rgb_composite = np.clip(rgb_composite, 0, 1)
    
    return rgb_composite

def plot_images(image_data, color, percentiles=(0,95)):
    magenta_cmap = AVAILABLE_COLORMAPS['magenta']
    green_cmap = AVAILABLE_COLORMAPS['green']
    bop_blue_cmap = AVAILABLE_COLORMAPS['bop blue']
    
    # sample many points in [0, 1] to build an RGBA LUT
    x = np.linspace(0, 1, 256)
    magenta_rgba = magenta_cmap.map(x)      # shape (256, 4)
    green_rgba = green_cmap.map(x)
    bop_blue_rgba = bop_blue_cmap.map(x)

    # wrap as Matplotlib colormaps
    magenta_mpl = ListedColormap(magenta_rgba, name='napari_magenta')
    green_mpl = ListedColormap(green_rgba, name='napari_green')
    bop_blue_mpl = ListedColormap(bop_blue_rgba, name='napari_bop_blue')
    
    if color == True:
        luts = [magenta_mpl, green_mpl, bop_blue_mpl]
    else:
        luts = ['gray', 'gray', 'gray']
    
    plt.figure(figsize=(12,12))
    nr_chan = image_data.shape[0]
    subplot_pos = 1
    for chan in range(nr_chan):
        if chan == 1:
            continue  # skip green channel for now
        plt.subplot(1, nr_chan, subplot_pos)
        plt.imshow(
            image_data[chan],
            cmap=luts[chan],
            vmin=np.percentile(image_data[chan].flatten(), percentiles[0]),
            vmax=np.percentile(image_data[chan].flatten(), percentiles[1]),
        )
        plt.axis('off')
        subplot_pos += 1
    rgb_image = create_rgb_composite(image_data, percentile_norm=percentiles)
    plt.subplot(1, nr_chan, subplot_pos)
    plt.imshow(rgb_image)
    plt.axis('off')
    plt.show()

def square_roi_coordinates(roi_1, roi_2, max_shape):
    max_image_size = list(max_shape[1:])
    roi_size = [0]*len(roi_2)
    for i in range(len(roi_2)):
        roi_size[i] = roi_2[i]-roi_1[i]
    # print(roi_size)

    roi_difference = max(roi_size[1], roi_size[2]) - min(roi_size[1], roi_size[2])
    # print("roi differente", roi_difference)

    new_roi_1 = roi_1.copy()
    new_roi_2 = roi_2.copy()

    if roi_size[1] > roi_size[2]: # if y is larger than x
        new_roi_1[2] -= roi_difference//2 # increase x by the difference
        new_roi_2[2] += roi_difference//2
    else: # if x is larger than y
        new_roi_1[1] -= roi_difference//2 # increase y by the difference
        new_roi_2[1] += roi_difference//2
    
    for i in range(len(new_roi_1)):
        if new_roi_1[i] < 0:
            new_roi_1[i] = 0
    for i in range(1, len(new_roi_2)):
        if new_roi_2[i] > max_image_size[i]:
            new_roi_2[i] = max_image_size[i]
    return new_roi_1, new_roi_2

# Reading data

Paste the path to the data below and store it as variable `str_path`.
When copying it from the site, the entire command should be:
```bash
napari --plugin napari-ome-zarr /path/to/data/in/hdd/
```
Paste the entire command to the code cell, delete everything except the napari call and replace the string in the variable `str_path`

In [ ]:
str_path = "/media/npmartins/ZS_HDD6/Data/Ex010_Re02/20250318/processed_files/NPM_Ex010_Re02_postExM_gel_b6_Im-01_AcquisitionBlock3_Airyscan_Ch.zarr"
data_path = Path(str_path)
file_name = "Stg234_Mito_9"

In [ ]:
zarr_file = parse_url(data_path, mode='r')
reader = Reader(zarr_file)
nodes = list(reader())
print(len(nodes)) # checking number of nodes in data. Should be = 1

## Getting file info

The next cell will print all the resolution levels present in the dataset. Choose the highest level one you can open in your computer.
Levels are zero (0) indexed, first one will be `reoslution_level = 0`

In [ ]:
zarr_info = info(data_path)
img_info = list(zarr_info)[0].data

# for item in img_info:
#     print(item.shape)

the next cell specifies the resolution level. Adjust depending on dataset and computer. Then run the following cell to load data.

In [ ]:
resolution_level = 1


In [ ]:
image_node = nodes[0]

dask_data = image_node.data
# print(len(dask_data))

image_array = dask_data[resolution_level].compute()
# image_array = dask_data[resolution_level]
print(image_array.shape)
image_array = image_array.squeeze()
print(image_array.shape)

In [ ]:
# image_node.metadata

In [ ]:
metadata = image_node.metadata
pixel_sizes = metadata['coordinateTransformations'][resolution_level][0]["scale"]
print(len(pixel_sizes))
print(pixel_sizes)

crop image stack to load only a stage chamber for inspection

In [ ]:
istart, iend = [2, 1177, 735], [132, 2152, 1871]

print("original image dimensions: "+str(image_array.shape))
image_array = crop_image(image_array, istart, iend)
image_array = image_array.compute()
print("cropped image dimensions: "+str(image_array.shape))

if loading and moving up and down `Z` is too slow, load the entire dataset into memory by uncommenting (removing # at each line) and runnning the following cell.

__Note__: this will need much more RAM!

In [ ]:
# image_node = nodes[0]

# dask_data = image_node.data
# # print(len(dask_data))

# image_array = dask_data[resolution_level]
# image_array = image_array.computer()
# print(image_array.shape)
# image_array = image_array.squeeze()
# print(image_array.shape)

## Load Napari

In [ ]:
import napari

In [ ]:
viewer = napari.Viewer()
layer1 = viewer.add_image(image_array[0], colormap='magenta', blending="additive")
layer2 = viewer.add_image(image_array[1], colormap='green', blending="additive")
layer3 = viewer.add_image(image_array[2], colormap='bop blue', blending="additive")

In [ ]:
# layer1_crop.scale = pixel_sizes[ -(len(layer1_crop.scale)) : ]
for layer in viewer.layers:
    viewer.layers[str(layer)].scale = pixel_sizes[ -(len(viewer.layers[str(layer)].scale)) : ]

if you need to adjust contrast, run the next cell: change the min and max values to whatever value is best.
Comment and uncomment the best option for you (remove the # from the beginning of the line)

In [ ]:
# for item in viewer.layers: viewer.layers[str(item)].contrast_limits=(0,600) # adjust B&C for all channels with the same value
viewer.layers[str(viewer.layers[0])].contrast_limits=(0, 700) # adjust B&C for first channel (pan-labeling)
# viewer.layers[str(viewer.layers[0])].contrast_limits=(0, 50) # adjust B&C for first channel (gfp)
viewer.layers[str(viewer.layers[2])].contrast_limits=(0, 150) # adjust B&C for first channel (nuclei)

In [ ]:
for i, item in enumerate(viewer.layers): 
    contrast_min, contrast_max = np.percentile(image_array[i].flatten(), 1), np.percentile(image_array[i].flatten(), 99)
    print(contrast_min, contrast_max)
    viewer.layers[str(item)].contrast_limits=(contrast_min, contrast_max)

### ROI image

In [ ]:
# roi_start = [1159, 429] #dims: y, x
# roi_start = [98, 303, 339] # dims: z, y, x
# # roi_end = [1454, 1001] #dims: y, x
# roi_end = [115, 445, 533] # dims: z, y, x
roi_start=[94, 308, 0]
roi_end=[129, 912, 485]

make ROI a square

In [ ]:
print(roi_start, roi_end)
roi_start_corrected, roi_end_corrected = square_roi_coordinates(roi_start, roi_end, image_array.shape)
print(roi_start_corrected, roi_end_corrected)

In [ ]:
# image_crop = image_array.compute()
# print(image_crop.shape)
image_crop = crop_image(image_array, roi_start_corrected, roi_end_corrected)
try:
    image_crop = image_crop.compute()
except:
    pass
print(image_crop.shape)

In [ ]:
viewer_crop = napari.Viewer()
layer1_crop = viewer_crop.add_image(image_crop[0], colormap='magenta', blending="additive")
layer2_crop = viewer_crop.add_image(image_crop[1], colormap='green', blending="additive")
layer3_crop = viewer_crop.add_image(image_crop[2], colormap='bop blue', blending="additive")

# layer1_crop.scale = pixel_sizes[ -(len(layer1_crop.scale)) : ]
for layer in viewer_crop.layers:
    viewer_crop.layers[str(layer)].scale = pixel_sizes[ -(len(viewer_crop.layers[str(layer)].scale)) : ]

In [ ]:
for item in viewer_crop.layers: viewer.layers[str(item)].contrast_limits=(0,1500)

Do a max projection

In [ ]:
max_projection = max_z_projection(image_crop)
print(max_projection.shape)

to plot a plane only

In [ ]:
max_projection = image_crop[..., image_crop.shape[1]//2, :, :]
# max_projection = image_crop[..., int(image_crop.shape[1]*0.6), :, :]
# max_projection = image_crop[..., 10, :, :]
# max_projection = image_array[..., image_array.shape[1]//2, :, :]

### Orthogonal view
one axis only

In [ ]:
print(image_crop.shape)
orthogonal_view = np.moveaxis(image_crop, 1, 2)
print(orthogonal_view.shape)

In [ ]:
print(pixel_sizes)
pixel_sizes_ortho = pixel_sizes.copy()
pixel_sizes_ortho[2], pixel_sizes_ortho[3] = pixel_sizes[3], pixel_sizes[2]
print(pixel_sizes_ortho)

In [ ]:
viewer_ortho = napari.Viewer()
layer1_ortho = viewer_ortho.add_image(orthogonal_view[0], colormap='magenta')
layer2_ortho = viewer_ortho.add_image(orthogonal_view[1], colormap='green')
layer3_ortho = viewer_ortho.add_image(orthogonal_view[2], colormap='bop blue')

for layer in viewer_ortho.layers:
    viewer_ortho.layers[str(layer)].scale = pixel_sizes_ortho[ -(len(viewer_ortho.layers[str(layer)].scale)) : ]

# Create figure from image

In [ ]:
# to remove green channel
# max_projection[1] = np.zeros_like(max_projection[1])
plot_images(max_projection, False, percentiles=(0, 99))

## Save data

In [ ]:
from tifffile import imwrite

In [ ]:
save_path = Path('/mnt/Data/nuno_martins/Image_data/ExM_processed_data/Share_data/'+str(file_name)+"/")
if not save_path.exists():
    save_path.mkdir(parents=True)

In [ ]:
print(save_path)

In [ ]:
# image = image_crop.compute()
image = image_crop
# image = image_array
image = np.clip(image, 0, 2**16-1)
image = image.astype(np.uint16)
image = np.moveaxis(image, 0, 1)
print(image.shape)

In [ ]:
image_name = 'follicle_cells'
imwrite(
    str(save_path)+"/"+file_name+"_"+image_name+"_resolution_lvl_"+str(resolution_level)+'.tif', 
    image, 
    dtype=np.uint16, 
    resolution=(1/float(pixel_sizes[3]), 1/float(pixel_sizes[4])), 
    metadata={
        'axes': 'ZCYX', 
        'PysicalSizeX' : pixel_sizes[4],
        'PysicalSizeXUnit' : 'um',
        'PysicalSizeY' : pixel_sizes[3],
        'PysicalSizeYUnit' : 'um',
        'PysicalSizeZ' : pixel_sizes[2],
        'PysicalSizeZUnit' : 'um',
        'unit':'um',
        }, 
    imagej=True, 
)

In [ ]:
pixel_sizes